# Qwen3 Assignment 3 benchmark evaluation

This notebook compares the pinned original Qwen3-1.7B model (C0) with the already frozen C1 configuration: original adjacent layers [3,4], Average operator, selected by normalized calibration-selection MSE. It does not run CGOS, choose a new pair, tune from benchmark scores, or train weights.

## Run instructions

1. Select a Tesla T4 Colab runtime. Place this notebook and the project companion files in one folder; set PROJECT_DIR in the next cell if needed.
2. Put the recovered C1 manifest, result JSON, and fitted affine tensors under c1_frozen_artifact/. These are reproducibly regenerated, hash-verified tensors; bit-for-bit identity with the earlier unsaved run is not claimed.
3. Run cells in order. The notebook validates the frozen C1 artifact before loading any benchmark test split. It saves predictions and errors as it runs.

## Dataset and prompt protocol

The available Assignment 3.docx is a general project rubric. It does not specify benchmark dataset revisions/configs, exact language coverage, few-shot IDs/counts, prompt wording, extraction/scoring, or token accounting. The 100,000-token target and few-shot requirement come from the project brief. The settings below are explicit, frozen protocol proposals, not claims about what the rubric mandates.

| Benchmark | Source/revision/split | Coverage | Few-shot source | Score |
|---|---|---|---|---|
| MMMLU | openai/MMMLU at 325a01dc3e173cac1578df94120499aaca2e2504, test | 14 locale configs: AR_XY, BN_BD, DE_DE, ES_LA, FR_FR, HI_IN, ID_ID, IT_IT, JA_JP, KO_KR, PT_BR, SW_KE, YO_NG, ZH_CN | This release provides test only. Use the five English MMLU development examples for the matching subject from cais/mmlu/all/dev at c30699e8356da336a370243923dbaf21066bb9fe. The same fixed subject examples are used for that subject in every translated locale. This is a language-mismatched but test-safe proposal. | Greedy generation; use the last matching Answer label found anywhere in the response (details below). |
| INCLUDE | Original CohereLabs/include-base-44 at ddecb994d8de145701d194108a80cb969984ba36, test | 41 of 44 configs; Urdu, Uzbek, and Vietnamese are explicitly excluded | First five examples in stable validation order for each evaluated config | Greedy generation; use the last matching Answer label found anywhere in the response; answer index 0-3 maps to A-D. |
| MGSM | juletxara/mgsm at 8db2cacc60a56941cc885a93d49c1330523a7c9b, test | en, es, fr, de, ru, zh, ja, th, sw, bn, te (English plus ten translated languages) | All eight official in-language train exemplars per config | Greedy generation; parse the number on the final Answer: line and compare numerically to answer_number. |

Five-shot MMMLU/INCLUDE, eight-shot MGSM, English-only MMMLU demos, prompt wording, output budgets, parsing, and token counting are proposed settings because the assignment does not define them. The original INCLUDE paper describes 44 languages and 5-shot prompting; the MGSM release provides eight translated train exemplars per language. Exact exemplar records, order, prompts and token hashes are saved. At the pinned INCLUDE revision, the schema uses either four `option_a`-`option_d` string columns or a `choices` sequence. Full validation-split inspection found zero complete choice rows in Urdu (0/15), Uzbek (0/20), and Vietnamese (0/15): every validation row has `choices=null`, and these configs have no alternative non-test split. They are excluded, so reported INCLUDE coverage is 41/44. The remaining 41 configs retain the same five-shot validation protocol. The current MCQ parser scans the full generated response case-insensitively for lines beginning with optional “Final ” then “Answer:”, accepts ASCII or full-width colon, takes the last A/B/C/D label match, and allows trailing text after that label. It does not require that match to be the final response line or the whole line. A stricter parser would be a separate proposed scoring change; it is not applied here.

## Token budget and fairness

Target 100,000 unique test-question payload tokens per dataset/language, using the pinned Qwen tokenizer. Payload means the test question plus answer choices when present; few-shot examples and chat-template overhead are excluded, and full prompted input tokens are also reported. Order is deterministic by SHA-256(seed 613, dataset, config, stable row ID); take unique questions without replacement through the first complete question reaching or crossing target. If a split is too small, use every available question once, report the shortfall, and never duplicate or pad. MGSM has a finite 250-question test split per language in this release.

C0 and C1 use identical prompt text, tokenizer, chat template with enable_thinking=False, token IDs, bfloat16, eager attention, greedy decoding, extraction/scoring, and use_cache=False. Per-example prompt and token-ID hashes are compared. C1 cannot reproduce attention, RoPE, or KV caching; disabling cache for both conditions provides matched inference settings, not functional equivalence.

Outputs under benchmark_results/<UTC run id>/ include the run manifest, exact few-shot examples, selected question IDs/token hashes, raw C0/C1 predictions, paired predictions, per-language summary, skipped examples, errors, and runtime log. No benchmark was run while creating this notebook.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, json, hashlib, sys

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "requirements.txt").exists():
    PROJECT_DIR = Path("/content/CS613_Assignment3_Qwen3")

MODEL_ID = "Qwen/Qwen3-1.7B"
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
PAIR = [3, 4]
OPERATOR = "Average"
SELECTION_METRIC = "normalized_mse"
SEED = 613
TOKEN_TARGET = 100_000
MCQ_MAX_NEW_TOKENS = 64
MGSM_MAX_NEW_TOKENS = 256
ATTN_IMPL = "eager"

MMMLU_ID = "openai/MMMLU"
MMMLU_REV = "325a01dc3e173cac1578df94120499aaca2e2504"
MMMLU_CONFIGS = ["AR_XY","BN_BD","DE_DE","ES_LA","FR_FR","HI_IN","ID_ID","IT_IT","JA_JP","KO_KR","PT_BR","SW_KE","YO_NG","ZH_CN"]
MMLU_DEV_ID = "cais/mmlu"
MMLU_DEV_REV = "c30699e8356da336a370243923dbaf21066bb9fe"
INCLUDE_ID = "CohereLabs/include-base-44"
INCLUDE_REV = "ddecb994d8de145701d194108a80cb969984ba36"
INCLUDE_ALL_CONFIGS = ["Albanian","Arabic","Armenian","Azerbaijani","Basque","Belarusian","Bengali","Bulgarian","Chinese","Croatian","Dutch","Estonian","Finnish","French","Georgian","German","Greek","Hebrew","Hindi","Hungarian","Indonesian","Italian","Japanese","Kazakh","Korean","Lithuanian","Malay","Malayalam","Nepali","North Macedonian","Persian","Polish","Portuguese","Russian","Serbian","Spanish","Tagalog","Tamil","Telugu","Turkish","Ukrainian","Urdu","Uzbek","Vietnamese"]
INCLUDE_EXCLUSION_REASON = "every validation row has choices=null and no alternative non-test split exists"
INCLUDE_EXCLUDED_CONFIGS = [
    {"config": name, "reason": INCLUDE_EXCLUSION_REASON}
    for name in ["Urdu", "Uzbek", "Vietnamese"]
]
INCLUDE_EXCLUDED_NAMES = {item["config"] for item in INCLUDE_EXCLUDED_CONFIGS}
INCLUDE_CONFIGS = [name for name in INCLUDE_ALL_CONFIGS if name not in INCLUDE_EXCLUDED_NAMES]
if len(INCLUDE_ALL_CONFIGS) != 44 or len(INCLUDE_CONFIGS) != 41 or len(INCLUDE_EXCLUDED_CONFIGS) != 3:
    raise RuntimeError("INCLUDE coverage must be 41/44 with exactly the three audited exclusions.")
INCLUDE_COVERAGE_SUMMARY = {
    "configured_config_count": len(INCLUDE_ALL_CONFIGS),
    "evaluated_config_count": len(INCLUDE_CONFIGS),
    "coverage": f"{len(INCLUDE_CONFIGS)}/{len(INCLUDE_ALL_CONFIGS)}",
    "excluded_configs": INCLUDE_EXCLUDED_CONFIGS,
}
MGSM_ID = "juletxara/mgsm"
MGSM_REV = "8db2cacc60a56941cc885a93d49c1330523a7c9b"
MGSM_CONFIGS = ["en","es","fr","de","ru","zh","ja","th","sw","bn","te"]

C1_DIR = PROJECT_DIR / "c1_frozen_artifact"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = PROJECT_DIR / "benchmark_results" / RUN_ID
RUN_DIR.mkdir(parents=True, exist_ok=False)
RUNTIME_LOG = RUN_DIR / "runtime_setup.log"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

def log(text):
    with RUNTIME_LOG.open("a",encoding="utf-8") as f:
        f.write(str(text).rstrip()+"\n")

print("Project:",PROJECT_DIR.resolve())
print("Run output:",RUN_DIR.resolve())
print("Frozen C1:",MODEL_ID,MODEL_REVISION,PAIR,OPERATOR,SELECTION_METRIC)

Project: /content
Run output: /content/benchmark_results/20261106T092234661371Z
Frozen C1: Qwen/Qwen3-1.7B 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e [3, 4] Average normalized_mse


In [ ]:
# Safety preflight: never install or replace PyTorch on a mismatched runtime.
import importlib.metadata as metadata
import re, subprocess, tempfile

required=["qwen3_cgos.py","requirements.txt","QWEN3_ARCHITECTURE_AUDIT.md","C1_PILOT_REPORT.md"]
missing=[x for x in required if not (PROJECT_DIR/x).is_file()]
if missing: raise FileNotFoundError(f"Set PROJECT_DIR to the project root; missing {missing}")
try:
    import torch
except Exception as exc:
    INSTALL_OK=False
    raise RuntimeError("Preinstalled PyTorch is required; no pip installation was attempted.") from exc

before={"torch":torch.__version__,"cuda":torch.version.cuda,"cuda_available":bool(torch.cuda.is_available())}
gpu=torch.cuda.get_device_name(0) if before["cuda_available"] else None
gpu_mem=torch.cuda.get_device_properties(0).total_memory/(1024**3) if gpu else None
print("Python:",sys.version.split()[0],"Pre-install PyTorch/CUDA:",before,"GPU:",gpu,"GPU memory GiB:",gpu_mem)
log(json.dumps({"python":sys.version.split()[0],"before":before,"gpu":gpu,"gpu_memory_gib":gpu_mem},sort_keys=True))
if before!={"torch":"2.11.0+cu130","cuda":"13.0","cuda_available":True} or "T4" not in str(gpu).upper():
    INSTALL_OK=False
    raise RuntimeError(f"Expected torch 2.11.0+cu130, CUDA 13.0, CUDA available, Tesla T4; detected {before}, GPU={gpu!r}. No pip install attempted.")

critical=re.compile(r"(?<![A-Za-z0-9_.-])(torch|transformers|datasets|accelerate|safetensors)(?![A-Za-z0-9_.-])",re.I)
def pip_check():
    p=subprocess.run([sys.executable,"-m","pip","check"],cwd=PROJECT_DIR,capture_output=True,text=True)
    out=((p.stdout or "")+("\n"+p.stderr if p.stderr else "")).strip()
    lines=[x for x in out.splitlines() if x.strip()]
    return p.returncode,out or "No dependency conflicts reported.",lines

base_code,base_out,base_lines=pip_check()
base_critical=[x for x in base_lines if critical.search(x)]
print("Pre-install pip check:",base_out); log("Pre-install pip check:\n"+base_out)
if base_critical:
    INSTALL_OK=False
    raise RuntimeError("Pre-existing conflicts involve critical packages; stopped before pip install: "+json.dumps(base_critical))

constraint=None
try:
    with tempfile.NamedTemporaryFile(mode="w",encoding="utf-8",suffix=".txt",delete=False) as f:
        f.write("torch=="+before["torch"]+"\n"); constraint=Path(f.name)
    proc=subprocess.run([sys.executable,"-m","pip","install","-r",str(PROJECT_DIR/"requirements.txt"),"--constraint",str(constraint)],cwd=PROJECT_DIR,capture_output=True,text=True)
finally:
    if constraint: constraint.unlink(missing_ok=True)
pip_output=(proc.stdout or "")+("\n--- stderr ---\n"+proc.stderr if proc.stderr else "")
print(pip_output); log("pip install exit="+str(proc.returncode)+"\n"+pip_output)
if proc.returncode:
    INSTALL_OK=False
    raise RuntimeError("Pinned requirements installation failed; see runtime_setup.log.")

probe=subprocess.run([sys.executable,"-c","import json,torch; print(json.dumps({'torch':torch.__version__,'cuda':torch.version.cuda,'cuda_available':bool(torch.cuda.is_available())}))"],cwd=PROJECT_DIR,capture_output=True,text=True)
if probe.returncode: raise RuntimeError("Fresh-process torch/CUDA probe failed: "+probe.stderr)
after=json.loads(probe.stdout.strip().splitlines()[-1])
if after!=before: raise RuntimeError(f"PyTorch/CUDA changed after dependency install: before={before}, after={after}")
pinned={"transformers":"4.51.0","datasets":"3.3.2","accelerate":"1.15.0","safetensors":"0.8.0","huggingface-hub":"0.36.2","tokenizers":"0.21.4","fsspec":"2024.12.0"}
actual={}
for name in pinned:
    try: actual[name]=metadata.version(name)
    except metadata.PackageNotFoundError: actual[name]="<missing>"
version_errors={k:{"expected":v,"actual":actual[k]} for k,v in pinned.items() if actual[k]!=v}
post_code,post_out,post_lines=pip_check()
post_critical=[x for x in post_lines if critical.search(x)]
unrelated_before=[x for x in base_lines if not critical.search(x)]
unrelated_after=[x for x in post_lines if not critical.search(x)]
conflicts={"preexisting_unrelated":unrelated_before,"still_present":[x for x in unrelated_after if x in unrelated_before],"new_unrelated":[x for x in unrelated_after if x not in unrelated_before],"resolved":[x for x in unrelated_before if x not in unrelated_after],"critical_after":post_critical}
print("Fresh PyTorch/CUDA:",after,"Pinned package versions:",actual)
print("Post-install pip check:",post_out,"\nConflict classification:",json.dumps(conflicts,indent=2))
log("Versions="+json.dumps(actual,sort_keys=True)+"\nPost-install pip check:\n"+post_out+"\nConflicts="+json.dumps(conflicts,sort_keys=True))
INSTALL_OK=not version_errors and not post_critical
if not INSTALL_OK: raise RuntimeError(f"Post-install verification failed: versions={version_errors}; critical conflicts={post_critical}")
print("INSTALL_OK =",INSTALL_OK)

Python: 3.13.16 Pre-install PyTorch/CUDA: {'torch': '2.11.0+cu130', 'cuda': '13.0', 'cuda_available': True} GPU: Tesla T4 GPU memory GiB: 14.56317138671875
Pre-install pip check: ipython 7.34.0 requires jedi, which is not installed.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 25.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 24.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 42.7 MB/s  0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.12.0
    Uninstalling fsspec-2025.12.0:
      Successfully uninstalled fsspec-2025.12.0
  Attempting uninstall: dill
    Found existing installation: dill 0.4.1
    Uninstalling dill-0.4.1:
      Successfully uninstalled dill-0.4.1
  Attempting uninstall: multiprocess
    Found existing installation: multiprocess 0.70.19
    Uninstalling multiprocess-0.70.19:
      Successfully uninstalled multiprocess-0.70.19
  Attempting uninstall: huggingface-hub
    Found

## Frozen C1 artifact gate

Before the benchmark reads test data, stage these files in `c1_frozen_artifact/`: `manifest.json`, the reproduced C1 result JSON, and `selected_affine.safetensors`. The recovered artifact records Qwen/Qwen3-1.7B at revision `70d244cc86ccca08cf5af4e1e306ecf908b1ad5e`, pair [3,4], Average, normalized-MSE selection, and SHA-256 hashes for the result and tensor files. The notebook validates these fields, hashes, tensor names, shapes, and finite values before loading benchmark data.

The affine tensors were reproducibly regenerated because the earlier revised run did not save them; their manifest explicitly does not claim bit-for-bit identity with that earlier run. This notebook reconstructs C1 from a fresh copy of the pinned original checkpoint and the verified tensors. It does not rerun CGOS.


In [ ]:

from pathlib import Path
import zipfile

zip_path = Path("/content/c1_frozen_artifact.zip")
artifact_dir = Path("/content/c1_frozen_artifact")

required = [
    "manifest.json",
    "c1_reproduced_results.json",
    "selected_affine.safetensors",
]

# Extract if the folder is missing or incomplete
if not all((artifact_dir / f).is_file() for f in required):
    if not zip_path.is_file():
        print("ZIP NOT FOUND. Files currently in /content:")
        print([p.name for p in Path("/content").iterdir()])
        raise FileNotFoundError(
            "Upload c1_frozen_artifact.zip to Colab Files, then rerun this cell."
        )

    with zipfile.ZipFile(zip_path) as z:
        names = set(z.namelist())
        missing = [f for f in required if f not in names]
        if missing:
            raise RuntimeError(f"ZIP is missing: {missing}")
        artifact_dir.mkdir(exist_ok=True)
        for f in required:
            with z.open(f) as src, open(artifact_dir / f, "wb") as dst:
                dst.write(src.read())

print("ARTIFACT FILE CHECK:")
for f in required:
    p = artifact_dir / f
    print(f, p.is_file(), p.stat().st_size if p.is_file() else "MISSING")

assert all((artifact_dir / f).is_file() for f in required)
print("All three artifact files are present.")


ARTIFACT FILE CHECK:
manifest.json True 2743
c1_reproduced_results.json True 125937
selected_affine.safetensors True 16785760
All three artifact files are present.


In [ ]:
import json
from safetensors.torch import load_file

def sha256_file(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for chunk in iter(lambda:f.read(1024*1024),b""): h.update(chunk)
    return h.hexdigest()

manifest_path=C1_DIR/"manifest.json"
if not manifest_path.is_file():
    raise FileNotFoundError(f"Frozen revised C1 artifact missing at {C1_DIR}. No benchmark split was loaded. Provide manifest, revised result JSON, and exact A/b state; the workspace historical result is rejected.")
m=json.loads(manifest_path.read_text(encoding="utf-8"))
expected={"artifact_format":"qwen3_c1_affine_v1","model_id":MODEL_ID,"model_revision":MODEL_REVISION,"tokenizer_revision":MODEL_REVISION,"pair_layers_zero_based":PAIR,"operator":OPERATOR,"selection_metric":SELECTION_METRIC,"fit_tokens":4096,"selection_tokens":1024}
bad={k:{"expected":v,"actual":m.get(k)} for k,v in expected.items() if m.get(k)!=v}
if bad: raise ValueError(f"C1 manifest mismatch: {bad}")
result_path=C1_DIR/m["results_file"]; state_path=C1_DIR/m["state_file"]
if not result_path.is_file() or not state_path.is_file(): raise FileNotFoundError("C1 result JSON/state file referenced by manifest is missing; test data not loaded.")
result_sha,state_sha=sha256_file(result_path),sha256_file(state_path)
if result_sha!=m.get("results_sha256") or state_sha!=m.get("state_sha256"): raise ValueError("C1 artifact SHA-256 mismatch; refusing benchmark.")
result=json.loads(result_path.read_text(encoding="utf-8")); selected=result.get("selected_c1",{})
if result.get("model_revision")!=MODEL_REVISION or selected.get("pair_layers_zero_based")!=PAIR or selected.get("operator")!=OPERATOR:
    raise ValueError(f"Revised C1 result does not verify pinned [3,4] Average: {selected}")
if "normalized" not in str(result.get("candidate_selection","")).lower() or len(result.get("candidate_rows",[]))!=81 or result.get("benchmark_data_used") is not False:
    raise ValueError("Expected normalized-MSE selection, 81 candidates, and no benchmark data in C1 result.")
if result.get("fit_token_budget")!=4096 or result.get("selection_token_budget")!=1024:
    raise ValueError("C1 fit/selection budgets do not match the frozen revised pilot.")
affine=load_file(str(state_path),device="cpu")
if set(affine)!={"A","b"} or tuple(affine["A"].shape)!=(2048,2048) or tuple(affine["b"].shape)!=(2048,):
    raise ValueError(f"Expected A[2048,2048], b[2048]; found {[(k,tuple(v.shape)) for k,v in affine.items()]}")
if not torch.isfinite(affine["A"]).all() or not torch.isfinite(affine["b"]).all(): raise FloatingPointError("Non-finite frozen affine coefficients.")
artifact_record={"manifest":m,"manifest_sha256":sha256_file(manifest_path),"results_sha256":result_sha,"state_sha256":state_sha,"selected_c1":selected}
(RUN_DIR/"frozen_c1_artifact_record.json").write_text(json.dumps(artifact_record,indent=2,sort_keys=True),encoding="utf-8")
print("Frozen C1 verified. No CGOS was run; no benchmark data was loaded.")

Frozen C1 verified. No CGOS was run; no benchmark data was loaded.


In [ ]:
# Tokenizer/config and deterministic prompt helpers. Test data is still unopened.
import gc, random, re, time, traceback
from collections import defaultdict
from decimal import Decimal, InvalidOperation
from collections.abc import Sequence
from numbers import Integral
import torch
from datasets import load_dataset
from huggingface_hub import snapshot_download
from transformers import AutoConfig, AutoTokenizer

random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic=True; torch.backends.cudnn.benchmark=False
try:
    TOKENIZER_SNAPSHOT_PATH = Path(snapshot_download(
        repo_id=MODEL_ID,
        revision=MODEL_REVISION,
        allow_patterns=[
            "config.json", "tokenizer.json", "tokenizer_config.json",
            "special_tokens_map.json", "added_tokens.json", "vocab.json",
            "merges.txt", "chat_template.jinja", "*.model",
        ],
    ))
except Exception as exc:
    raise RuntimeError(
        f"Could not resolve tokenizer snapshot for {MODEL_ID}@{MODEL_REVISION}; "
        "refusing to load a tokenizer without verified revision provenance."
    ) from exc
TOKENIZER_RESOLVED_COMMIT = TOKENIZER_SNAPSHOT_PATH.name
if TOKENIZER_SNAPSHOT_PATH.parent.name != "snapshots" or TOKENIZER_RESOLVED_COMMIT != MODEL_REVISION:
    raise RuntimeError(
        f"Tokenizer snapshot verification failed: requested {MODEL_REVISION}, "
        f"resolved path {TOKENIZER_SNAPSHOT_PATH} (commit {TOKENIZER_RESOLVED_COMMIT!r})."
    )
tokenizer=AutoTokenizer.from_pretrained(
    str(TOKENIZER_SNAPSHOT_PATH), local_files_only=True, use_fast=True, trust_remote_code=False
)
config=AutoConfig.from_pretrained(MODEL_ID,revision=MODEL_REVISION,trust_remote_code=False)
if int(config.num_hidden_layers)!=28 or int(config.hidden_size)!=2048: raise RuntimeError("Checkpoint architecture differs from audit.")
context_limit=int(config.max_position_embeddings)
if tokenizer.eos_token_id is None: raise RuntimeError("Tokenizer EOS token is missing.")

def encode_chat(prompt):
    enc=tokenizer.apply_chat_template([{"role":"user","content":prompt}],tokenize=True,add_generation_prompt=True,enable_thinking=False,return_dict=True,return_tensors="pt")
    ids=enc["input_ids"]; mask=enc.get("attention_mask",torch.ones_like(ids))
    return ids,mask
def digest_ids(ids):
    return hashlib.sha256(",".join(map(str,ids.reshape(-1).tolist())).encode("ascii")).hexdigest()
def digest_text(s): return hashlib.sha256(s.encode("utf-8")).hexdigest()
def choice_label(x, *, context="answer"):
    """Map documented answer labels or zero-based 0..3 indices to A..D."""
    if isinstance(x, bool):
        raise ValueError(f"{context}: boolean is not a valid multiple-choice answer: {x!r}")
    if isinstance(x, str):
        value = x.strip().upper()
        if value in {"A", "B", "C", "D"}:
            return value
        if value in {"0", "1", "2", "3"}:
            return "ABCD"[int(value)]
        raise ValueError(f"{context}: expected answer label A-D or zero-based index 0-3; got {x!r}")
    if isinstance(x, Integral):
        index = int(x)
        if 0 <= index < 4:
            return "ABCD"[index]
        raise ValueError(f"{context}: zero-based answer index must be in [0,3]; got {x!r}")
    raise ValueError(f"{context}: unsupported answer representation {type(x).__name__}: {x!r}")

def choices(row, *, context="row"):
    """Read four choices from a documented choices sequence or option_a..option_d."""
    keys = set(row.keys())
    raw = row.get("choices")
    if raw is not None:
        if not isinstance(raw, Sequence) or isinstance(raw, (str, bytes, bytearray)):
            raise ValueError(
                f"{context}: `choices` must be a sequence of four values; "
                f"got {type(raw).__name__} {raw!r}; columns={sorted(keys)}"
            )
        values = list(raw)
        source = "choices"
    else:
        lower = ["option_a", "option_b", "option_c", "option_d"]
        upper = ["A", "B", "C", "D"]
        if all(key in keys for key in lower):
            values = [row.get(key) for key in lower]
            source = "option_a..option_d"
        elif all(key in keys for key in upper):
            values = [row.get(key) for key in upper]
            source = "A..D"
        else:
            observed = {key: row.get(key) for key in ["choices", *lower, *upper] if key in keys}
            raise ValueError(
                f"{context}: no complete supported choice schema; expected `choices` sequence "
                f"or four option_a..option_d columns; columns={sorted(keys)}; observed={observed!r}"
            )
    if len(values) != 4:
        raise ValueError(
            f"{context}: {source} contains {len(values)} choices; exactly four are required; "
            f"values={values!r}; columns={sorted(keys)}"
        )
    missing = [i for i, value in enumerate(values) if value is None or (isinstance(value, str) and not value.strip())]
    if missing:
        raise ValueError(
            f"{context}: {source} has missing/blank choice values at zero-based positions {missing}; "
            f"values={values!r}; columns={sorted(keys)}"
        )
    if any(isinstance(value, (dict, list, tuple)) for value in values):
        raise ValueError(
            f"{context}: {source} contains nested non-text choices; expected scalar option text; "
            f"values={values!r}; columns={sorted(keys)}"
        )
    return [str(value) for value in values]

def mcq_block(q,ch,answer=None):
    s="Question: "+str(q)+"\n"+"\n".join(f"{l}. {c}" for l,c in zip("ABCD",ch))
    return s+(f"\nAnswer: {answer}" if answer else "")
def mcq_prompt(demos,q,ch):
    instruction="Answer each multiple-choice item. Choose one of A, B, C, or D. End with a final line exactly in the form Answer: X."
    return instruction+"\n\n"+"\n\n".join(mcq_block(d["question"],d["choices"],d["answer"]) for d in demos)+"\n\n"+mcq_block(q,ch)+"\nAnswer:"
def mgsm_prompt(demos,q):
    instruction="Solve each arithmetic word problem. Show a concise solution and end with a final line exactly in the form Answer: <number>."
    ex=[f"Question: {d['question']}\n{d['answer']}\nAnswer: {d['answer_number']}" for d in demos]
    return instruction+"\n\n"+"\n\n".join(ex+[f"Question: {q}\nSolution:\nAnswer:"])
def stable_key(dataset,config,example_id):
    return hashlib.sha256(f"{SEED}|{dataset}|{config}|{example_id}".encode()).hexdigest()

# English MMLU dev contains five examples per each of 57 subjects.
mmlu_dev=load_dataset(MMLU_DEV_ID,"all",split="dev",revision=MMLU_DEV_REV)
dev_by_subject=defaultdict(list)
for row in mmlu_dev:
    s=str(row["subject"]); n=len(dev_by_subject[s])
    dev_by_subject[s].append({"id":f"MMLU-dev:{s}:{n}","question":str(row["question"]),"choices":choices(row),"answer":choice_label(row["answer"])})
if len(mmlu_dev)!=285 or len(dev_by_subject)!=57 or any(len(v)<5 for v in dev_by_subject.values()):
    raise RuntimeError("Unexpected pinned English MMLU dev few-shot inventory.")
print("Pinned tokenizer ready; requested revision:",MODEL_REVISION,"verified snapshot commit:",TOKENIZER_RESOLVED_COMMIT,"context limit:",context_limit,"MMLU dev exemplars:",len(mmlu_dev))

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 5 files:   0%|          | 0/5 [00:00<?, ?it/s]

merges.txt: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

... (8 outputs left out)

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

Pinned tokenizer ready; requested revision: 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e verified snapshot commit: 70d244cc86ccca08cf5af4e1e306ecf908b1ad5e context limit: 40960 MMLU dev exemplars: 285


In [ ]:
import csv
import importlib.util
from huggingface_hub import hf_hub_download

# Download the official MGSM exemplar file at the already-pinned revision.
exemplars_path = hf_hub_download(
    repo_id=MGSM_ID,
    filename="exemplars.py",
    repo_type="dataset",
    revision=MGSM_REV,
)

spec = importlib.util.spec_from_file_location(
    "mgsm_exemplars_pinned", exemplars_path
)
mgsm_exemplars = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mgsm_exemplars)

print("MGSM exemplars loaded for:", len(mgsm_exemplars.MGSM_EXEMPLARS), "languages")

exemplars.py: 0.00B [00:00, ?B/s]

MGSM exemplars loaded for: 11 languages


In [ ]:
# Load non-test exemplars and then prepare deterministic unique test-question subsets.
tasks=[]; fewshot_manifest=[]; skipped=[]
def add_task(dataset,config_name,rows,demos,normalizer,task_type):
    fewshot_manifest.append({"dataset":dataset,"config":config_name,"shot_count":len(demos),"exemplars":demos})
    candidates=[]; invalid=[]
    for i,row in enumerate(rows):
        try:
            r=normalizer(config_name,i,row)
            r["payload_tokens"]=len(tokenizer.encode(r["payload"],add_special_tokens=False))
            r["fewshot_ids"]=[d["id"] for d in demos]
            r["prompt"]=mcq_prompt(demos,r["question"],r["choices"]) if task_type=="mcq" else mgsm_prompt(demos,r["question"])
            r["max_new_tokens"]=MCQ_MAX_NEW_TOKENS if task_type=="mcq" else MGSM_MAX_NEW_TOKENS
            r["prompt_sha256"]=digest_text(r["prompt"])
            ids,_=encode_chat(r["prompt"]); r["input_tokens"]=int(ids.shape[1]); r["input_ids_sha256"]=digest_ids(ids)
            if r["input_tokens"]+r["max_new_tokens"]>context_limit:
                invalid.append({"dataset":dataset,"config":config_name,"example_id":r["example_id"],"reason":"prompt plus output budget exceeds model context"}); continue
            candidates.append(r)
        except Exception as exc:
            invalid.append({"dataset":dataset,"config":config_name,"example_id":f"{dataset}:{config_name}:test:{i}","reason":f"{type(exc).__name__}: {exc}"})
    candidates.sort(key=lambda x:stable_key(dataset,config_name,x["example_id"]))
    selected=[]; n_tokens=0
    if dataset=="MGSM":
        selected=list(candidates); n_tokens=sum(r["payload_tokens"] for r in selected)
    else:
        for r in candidates:
            selected.append(r); n_tokens+=r["payload_tokens"]
            if n_tokens>=TOKEN_TARGET: break
    skipped.extend(invalid)
    tasks.append({"dataset":dataset,"config":config_name,"source_test_rows":len(rows),"valid_candidates":len(candidates),"selected":selected,"selected_questions":len(selected),"payload_tokens":n_tokens,"target_met":n_tokens>=TOKEN_TARGET,"shortfall_tokens":max(0,TOKEN_TARGET-n_tokens),"skipped_count":len(invalid)})
def norm_mmmlu(c, i, r):
    q = str(r["Question"])
    ch = [str(r[k]) for k in "ABCD"]
    gold = choice_label(r["Answer"])

    raw_subject = str(r["Subject"]).strip()

    # Fix malformed subject labels in MMMLU metadata.
    subject_aliases = {
        "college_mathematics_test.csv_sw-KE.csv":
            "college_mathematics",
        "security_studies_test-sw-KE.csv":
            "security_studies",
        "college_mathematics_test.csv_YO-NG.csv":
            "college_mathematics",
    }

    subject = subject_aliases.get(raw_subject, raw_subject)

    if subject not in dev_by_subject:
        raise ValueError(
            f"No subject-matched English MMLU dev examples: "
            f"raw={raw_subject!r}, normalized={subject!r}"
        )

    demos = dev_by_subject[subject][:5]

    out = {
        "dataset": "MMMLU",
        "config": c,
        "example_id": f"MMMLU:{c}:test:{i}",
        "question": q,
        "choices": ch,
        "gold": gold,
        "subject": subject,
        "payload": (
            "Question: " + q + "\n" +
            "\n".join(f"{l}. {x}" for l, x in zip("ABCD", ch))
        ),
        "fewshot_ids": [d["id"] for d in demos],
        "_demos": demos,
    }

    out["prompt"] = mcq_prompt(demos, q, ch)
    return out

def norm_include(c,i,r):
    q=str(r["question"]); ch=choices(r); gold=choice_label(r["answer"])
    return {"dataset":"INCLUDE","config":c,"example_id":f"INCLUDE:{c}:test:{i}","question":q,"choices":ch,"gold":gold,"payload":"Question: "+q+"\n"+"\n".join(f"{l}. {x}" for l,x in zip("ABCD",ch))}
def norm_mgsm(c,i,r):
    q=str(r["question"]); gold=str(r["answer_number"])
    return {"dataset":"MGSM","config":c,"example_id":f"MGSM:{c}:test:{i}","question":q,"gold":gold,"payload":"Question: "+q}

# Validate the frozen INCLUDE few-shot inventory before loading any benchmark test split.
INCLUDE_DEMOS_BY_CONFIG = {}
INCLUDE_PREFLIGHT_ROWS = []
INCLUDE_PREFLIGHT_ERRORS = []
for include_config in INCLUDE_CONFIGS:
    validation = load_dataset(
        INCLUDE_ID, include_config, split="validation", revision=INCLUDE_REV
    )
    columns = list(validation.column_names)
    if len(validation) < 5:
        INCLUDE_PREFLIGHT_ERRORS.append({
            "config": include_config,
            "split": "validation",
            "row_index": None,
            "error": f"five-shot protocol requires 5 rows; pinned split has {len(validation)}",
            "columns": columns,
        })
    demos = []
    row_status = []
    for row_index in range(min(5, len(validation))):
        row = validation[row_index]
        context = f"INCLUDE config={include_config!r} split='validation' row_index={row_index}"
        try:
            question = row.get("question")
            if not isinstance(question, str) or not question.strip():
                raise ValueError(f"{context}: missing or blank question")
            option_values = choices(row, context=context)
            answer = choice_label(row.get("answer"), context=context + " answer")
            demos.append({
                "id": f"INCLUDE:{include_config}:validation:{row_index}",
                "question": question,
                "choices": option_values,
                "answer": answer,
            })
            row_status.append({"row_index": row_index, "valid": True})
        except ValueError as exc:
            choice_fields = {key: row.get(key) for key in ["choices", "option_a", "option_b", "option_c", "option_d", "answer"] if key in row}
            INCLUDE_PREFLIGHT_ERRORS.append({
                "config": include_config,
                "split": "validation",
                "row_index": row_index,
                "columns": columns,
                "question": row.get("question"),
                "raw_choice_and_answer_fields": choice_fields,
                "error": str(exc),
            })
            row_status.append({"row_index": row_index, "valid": False, "error": str(exc)})
    schema = (
        "choices_sequence" if "choices" in columns else
        "option_a..option_d" if all(k in columns for k in ["option_a", "option_b", "option_c", "option_d"]) else
        "unrecognized"
    )
    INCLUDE_PREFLIGHT_ROWS.append({
        "config": include_config,
        "columns": columns,
        "choice_schema": schema,
        "validation_rows": len(validation),
        "five_shot_rows_checked": len(row_status),
        "valid_examples": sum(bool(item["valid"]) for item in row_status),
        "rows": row_status,
    })
    if len(demos) == 5:
        INCLUDE_DEMOS_BY_CONFIG[include_config] = demos

include_preflight = {
    "dataset": INCLUDE_ID,
    "revision": INCLUDE_REV,
    "configured_config_count": len(INCLUDE_ALL_CONFIGS),
    "evaluated_config_count": len(INCLUDE_CONFIGS),
    "coverage": f"{len(INCLUDE_CONFIGS)}/{len(INCLUDE_ALL_CONFIGS)}",
    "excluded_configs": INCLUDE_EXCLUDED_CONFIGS,
    "configs_checked": len(INCLUDE_PREFLIGHT_ROWS),
    "examples_checked": sum(item["five_shot_rows_checked"] for item in INCLUDE_PREFLIGHT_ROWS),
    "valid_examples": sum(item["valid_examples"] for item in INCLUDE_PREFLIGHT_ROWS),
    "errors": INCLUDE_PREFLIGHT_ERRORS,
    "configs": INCLUDE_PREFLIGHT_ROWS,
}
(RUN_DIR / "include_validation_preflight.json").write_text(
    json.dumps(include_preflight, ensure_ascii=False, indent=2), encoding="utf-8"
)
print("INCLUDE validation few-shot preflight:", json.dumps({k: include_preflight[k] for k in ["dataset", "revision", "configs_checked", "examples_checked", "valid_examples"]}, ensure_ascii=False))
print("INCLUDE coverage:", INCLUDE_COVERAGE_SUMMARY["coverage"], "configured; excluded:", [item["config"] for item in INCLUDE_EXCLUDED_CONFIGS])
if INCLUDE_PREFLIGHT_ERRORS:
    print("Malformed/missing INCLUDE few-shot rows; no examples were replaced:")
    print(json.dumps(INCLUDE_PREFLIGHT_ERRORS, ensure_ascii=False, indent=2))
    raise RuntimeError(
        f"INCLUDE five-shot preflight failed for {len(INCLUDE_PREFLIGHT_ERRORS)} rows across "
        f"{len({item['config'] for item in INCLUDE_PREFLIGHT_ERRORS})} configs. "
        "No benchmark test split was loaded by this cell. Missing choices cannot be reconstructed; "
        "resolve the source/protocol before resuming."
    )

for c in MMMLU_CONFIGS:
    test=list(load_dataset(MMMLU_ID,c,split="test",revision=MMMLU_REV))
    # MMMLU examples vary by subject; construct prompts with each subject's fixed five demos.
    candidates=[]; invalid=[]
    for i,row in enumerate(test):
        try:
            r=norm_mmmlu(c,i,row); r["payload_tokens"]=len(tokenizer.encode(r["payload"],add_special_tokens=False)); r["max_new_tokens"]=MCQ_MAX_NEW_TOKENS
            r["prompt_sha256"]=digest_text(r["prompt"]); ids,_=encode_chat(r["prompt"]); r["input_tokens"]=int(ids.shape[1]); r["input_ids_sha256"]=digest_ids(ids)
            if r["input_tokens"]+r["max_new_tokens"]>context_limit: invalid.append({"dataset":"MMMLU","config":c,"example_id":r["example_id"],"reason":"prompt plus output budget exceeds model context"}); continue
            candidates.append(r)
        except Exception as exc: invalid.append({"dataset":"MMMLU","config":c,"example_id":f"MMMLU:{c}:test:{i}","reason":f"{type(exc).__name__}: {exc}"})
    candidates.sort(key=lambda x:stable_key("MMMLU",c,x["example_id"]))
    selected=[]; n=0
    for r in candidates:
        selected.append(r); n+=r["payload_tokens"]
        if n>=TOKEN_TARGET: break
    skipped.extend(invalid)
    tasks.append({"dataset":"MMMLU","config":c,"source_test_rows":len(test),"valid_candidates":len(candidates),"selected":selected,"selected_questions":len(selected),"payload_tokens":n,"target_met":n>=TOKEN_TARGET,"shortfall_tokens":max(0,TOKEN_TARGET-n),"skipped_count":len(invalid)})
    used={d["id"]:d for r in selected for d in r["_demos"]}
    fewshot_manifest.append({"dataset":"MMMLU","config":c,"shot_count":5,"exemplars_by_subject":list(used.values())})

for c in INCLUDE_CONFIGS:
    demos = INCLUDE_DEMOS_BY_CONFIG[c]
    test=list(load_dataset(INCLUDE_ID,c,split="test",revision=INCLUDE_REV))
    add_task("INCLUDE",c,test,demos,norm_include,"mcq")

for c in MGSM_CONFIGS:
    train = [
        {
            "question": item["q"],
            "answer": item["a"],
            "answer_number": mgsm_exemplars.EXEMPLAR_NUMBER_ANSWERS[int(key) - 1],
        }
        for key, item in mgsm_exemplars.MGSM_EXEMPLARS[c].items()
    ]
    if len(train)!=8: raise RuntimeError(f"MGSM {c}: expected 8 train exemplars, got {len(train)}")
    demos=[{"id":f"MGSM:{c}:train:{i}","question":str(r["question"]),"answer":str(r["answer"]),"answer_number":str(r["answer_number"])} for i,r in enumerate(train)]
    test_path = hf_hub_download(
        repo_id=MGSM_ID,
        filename=f"mgsm_{c}.tsv",
        repo_type="dataset",
        revision=MGSM_REV,
    )

    test = []
    with open(test_path, "r", encoding="utf-8") as f:
        reader = csv.reader(
            f,
            quotechar='"',
            delimiter="\t",
            quoting=csv.QUOTE_ALL,
            skipinitialspace=True,
        )
        for row in reader:
            if len(row) < 2:
                raise RuntimeError(f"Malformed MGSM row in {c}: {row}")
            test.append({
                "question": row[0],
                "answer": None,
                "answer_number": int(row[1].replace(",", "")),
            })

    if len(test) != 250:
        raise RuntimeError(f"MGSM {c}: expected 250 test questions, got {len(test)}")
    add_task("MGSM",c,test,demos,norm_mgsm,"mgsm")

expected_task_count = len(MMMLU_CONFIGS) + len(INCLUDE_CONFIGS) + len(MGSM_CONFIGS)
if len(tasks) != expected_task_count: raise RuntimeError(f"Expected {expected_task_count} dataset-language configs, got {len(tasks)}")
if any(len({r["example_id"] for r in t["selected"]})!=len(t["selected"]) for t in tasks): raise RuntimeError("Duplicate selected IDs.")
(RUN_DIR/"fewshot_exemplars.json").write_text(json.dumps(fewshot_manifest,ensure_ascii=False,indent=2),encoding="utf-8")
selection=[{k:v for k,v in t.items() if k!="selected"}|{"selected_ids":[r["example_id"] for r in t["selected"]],"payload_tokens":[r["payload_tokens"] for r in t["selected"]],"prompt_sha256":[r["prompt_sha256"] for r in t["selected"]],"input_ids_sha256":[r["input_ids_sha256"] for r in t["selected"]]} for t in tasks]
(RUN_DIR/"selected_questions.json").write_text(json.dumps(selection,ensure_ascii=False,indent=2),encoding="utf-8")
with (RUN_DIR/"skipped_examples.jsonl").open("w",encoding="utf-8") as f:
    for r in skipped: f.write(json.dumps(r,ensure_ascii=False)+"\n")
print("Prepared language configs:",len(tasks),"selected test questions:",sum(t["selected_questions"] for t in tasks),"pre-scoring skips:",len(skipped),"INCLUDE coverage:",INCLUDE_COVERAGE_SUMMARY["coverage"])

README.md: 0.00B [00:00, ?B/s]

Albanian/test-00000-of-00001.parquet:   0%|          | 0.00/99.1k [00:00<?, ?B/s]

Albanian/validation-00000-of-00001.parqu(…):   0%|          | 0.00/10.5k [00:00<?, ?B/s]

Generating test split:   0%|          | 0/551 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/25 [00:00<?, ? examples/s]

Arabic/test-00000-of-00001.parquet:   0%|          | 0.00/115k [00:00<?, ?B/s]

... (197 outputs left out)

mgsm_sw.tsv: 0.00B [00:00, ?B/s]

mgsm_bn.tsv: 0.00B [00:00, ?B/s]

mgsm_te.tsv: 0.00B [00:00, ?B/s]

Prepared language configs: 66 selected test questions: 31220 pre-scoring skips: 0 INCLUDE coverage: 41/44


In [ ]:
from collections import Counter

print("Skip reasons:")
print(Counter(x["reason"] for x in skipped))

print("\nSelected questions by dataset:")
for dataset in ["MMMLU", "INCLUDE", "MGSM"]:
    subset = [t for t in tasks if t["dataset"] == dataset]
    print(
        dataset,
        "| configs:", len(subset),
        "| selected:", sum(t["selected_questions"] for t in subset),
        "| payload tokens:", sum(t["payload_tokens"] for t in subset),
    )

Skip reasons:
Counter()

Selected questions by dataset:
MMMLU | configs: 14 | selected: 8178 | payload tokens: 1402028
INCLUDE | configs: 41 | selected: 20292 | payload tokens: 2713393
MGSM | configs: 11 | selected: 2750 | payload tokens: 360457


In [ ]:
print("Total skipped:", len(skipped))

for i, item in enumerate(skipped[:10]):
    print(f"\nSkip {i+1}:")
    print(item)

print("\nSkip counts by dataset:")
from collections import Counter
print(Counter(x.get("dataset", "UNKNOWN") for x in skipped))

print("\nSkip counts by language:")
print(Counter(x.get("language", "UNKNOWN") for x in skipped))

Total skipped: 0

Skip counts by dataset:
Counter()

Skip counts by language:
Counter()


In [ ]:
from collections import Counter

print("Skipped MMMLU configurations:")
print(Counter(x["config"] for x in skipped))

print("\nUnique skipped example IDs by configuration:")
for config in sorted(set(x["config"] for x in skipped)):
    ids = [x["example_id"] for x in skipped if x["config"] == config]
    print(config, "| count:", len(ids), "| first:", ids[0], "| last:", ids[-1])

Skipped MMMLU configurations:
Counter()

Unique skipped example IDs by configuration:


In [ ]:
print("=== English MMLU dev dictionary ===")
print("Number of subjects:", len(dev_by_subject))
print("First 20 subject keys:")
print(list(dev_by_subject.keys())[:20])

for config in ["SW_KE", "YO_NG"]:
    print(f"\n=== MMMLU test config: {config} ===")

    ds = load_dataset(
        MMMLU_ID,
        config,
        split="test",
        revision=MMMLU_REV
    )

    print("Columns:", ds.column_names)
    subjects = Counter(str(row["Subject"]) for row in ds)
    print("Unique subjects:", len(subjects))
    print("Subject names and counts:")

    for subject, count in subjects.items():
        print(repr(subject), "| questions:", count,
              "| exact English key match:", subject in dev_by_subject)

=== English MMLU dev dictionary ===
Number of subjects: 57
First 20 subject keys:
['abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge', 'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics', 'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic', 'global_facts', 'high_school_biology', 'high_school_chemistry']

=== MMMLU test config: SW_KE ===
Columns: ['Unnamed: 0', 'Question', 'A', 'B', 'C', 'D', 'Answer', 'Subject']
Unique subjects: 57
Subject names and counts:
'abstract_algebra' | questions: 100 | exact English key match: True
'anatomy' | questions: 135 | exact English key match: True
'astronomy' | questions: 152 | exact English key match: True
'business_ethics' | questions: 100 | exact English key match: True
'clinical_knowledge' | questions: 265 | exact English key match: True
'college_biology' | questions

In [ ]:
subject_aliases_check = {
    "college_mathematics_test.csv_sw-KE.csv": "college_mathematics",
    "security_studies_test-sw-KE.csv": "security_studies",
    "college_mathematics_test.csv_YO-NG.csv": "college_mathematics",
}

print("MMMLU subject alias verification:\n")

for raw, expected in subject_aliases_check.items():
    actual = subject_aliases_check.get(raw, raw)

    print("Raw:", raw)
    print("Normalized:", actual)
    print("Exists in English MMLU:", actual in dev_by_subject)
    print()

    assert actual == expected
    assert actual in dev_by_subject

print("All three subject aliases verified successfully.")

MMMLU subject alias verification:

Raw: college_mathematics_test.csv_sw-KE.csv
Normalized: college_mathematics
Exists in English MMLU: True

Raw: security_studies_test-sw-KE.csv
Normalized: security_studies
Exists in English MMLU: True

Raw: college_mathematics_test.csv_YO-NG.csv
Normalized: college_mathematics
Exists in English MMLU: True

All three subject aliases verified successfully.


## Run matched model inference

The next cell performs the actual benchmark. C0 and C1 are loaded separately from the same pinned source checkpoint. C1 uses only the frozen A/b tensors after validating the pair/operator/selection record. Errors are retained as incorrect for accuracy; context/invalid-row skips are reported separately. This cell has not been executed as part of notebook creation.

In [ ]:
import gc
import torch

gc.collect()
torch.cuda.empty_cache()

print("GPU memory allocated:",
      round(torch.cuda.memory_allocated() / 1e9, 2), "GB")

GPU memory allocated: 0.0 GB


In [ ]:

# ============================================================
# MATCHED GREEDY INFERENCE: C0 vs C1
# C1 is rebuilt from the pinned source checkpoint and
# verified frozen A/b tensors.
#
# Frozen configuration:
#   Pair: [3, 4] (zero-based)
#   Operator: Average
#   Selection metric: normalized_mse
#   Precision: FP16 (Tesla T4 compatibility)
#   Decoding: Greedy
#   Initial run: 2-example smoke test
# ============================================================

from collections import defaultdict
from transformers import AutoModelForCausalLM, AutoConfig


# ------------------------------------------------------------
# 1. Validate frozen benchmark configuration
# ------------------------------------------------------------

if PAIR != [3, 4] or OPERATOR != "Average":
    raise RuntimeError(
        f"This notebook is frozen to pair [3,4]/Average; "
        f"found {PAIR}/{OPERATOR}."
    )

selected_artifact = artifact_record["selected_c1"]

if (
    selected_artifact.get("pair_layers_zero_based") != PAIR
    or selected_artifact.get("operator") != OPERATOR
):
    raise RuntimeError(
        "The verified C1 artifact does not match the frozen "
        "[3,4] Average configuration."
    )

if artifact_record["manifest"].get("selection_metric") != SELECTION_METRIC:
    raise RuntimeError(
        "The verified C1 artifact selection metric differs "
        "from this notebook protocol."
    )


# ------------------------------------------------------------
# 2. Load the correct model architecture configuration
# ------------------------------------------------------------

# IMPORTANT:
# Do not use the variable name `config` here.
# In this notebook, `config` may already be a dataset string.

model_config = AutoConfig.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    trust_remote_code=False,
)

if not hasattr(model_config, "hidden_size"):
    raise RuntimeError(
        "The pinned model configuration does not contain hidden_size."
    )

print("Model configuration:", type(model_config).__name__)
print("Model hidden size:", model_config.hidden_size)


# ------------------------------------------------------------
# 3. Import audited parameter-count and surgery functions
# ------------------------------------------------------------

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from qwen3_cgos import (
    EXPECTED_BLOCK_PARAMS,
    EXPECTED_TOTAL_PARAMS,
    _replace_pair,
)

C0_PARAMS = EXPECTED_TOTAL_PARAMS

C1_PARAMS = (
    C0_PARAMS
    - 2 * EXPECTED_BLOCK_PARAMS
    + int(model_config.hidden_size) ** 2
    + int(model_config.hidden_size)
)

if C0_PARAMS != 1_720_574_976 or C1_PARAMS != 1_624_099_328:
    raise RuntimeError(
        f"Unexpected audited parameter counts: "
        f"C0={C0_PARAMS:,}, C1={C1_PARAMS:,}."
    )

print(f"Expected C0 parameters: {C0_PARAMS:,}")
print(f"Expected C1 parameters: {C1_PARAMS:,}")


def nparams(model):
    return sum(parameter.numel() for parameter in model.parameters())


# ------------------------------------------------------------
# 4. Freeze selected examples for both model conditions
# ------------------------------------------------------------

SELECTED_EXAMPLES = [
    (task, row)
    for task in tasks
    for row in task["selected"]
]

SELECTED_IDS = [
    row["example_id"]
    for _, row in SELECTED_EXAMPLES
]

if len(SELECTED_IDS) != len(set(SELECTED_IDS)):
    raise RuntimeError(
        "Selected benchmark records contain duplicate example IDs."
    )

if not SELECTED_EXAMPLES:
    raise RuntimeError(
        "Dataset preparation selected no examples; "
        "refusing empty inference files."
    )

for task, row in SELECTED_EXAMPLES:

    if row["prompt_sha256"] != digest_text(row["prompt"]):
        raise RuntimeError(
            f"Stored prompt hash mismatch: {row['example_id']}"
        )

    if (
        row["input_tokens"] <= 0
        or row["input_tokens"] + row["max_new_tokens"] > context_limit
    ):
        raise RuntimeError(
            f"Invalid prepared token budget: {row['example_id']}"
        )


# ------------------------------------------------------------
# 5. Smoke-test mode
# ------------------------------------------------------------

# Keep True for the first run.
# Change to False ONLY after both C0 and C1 smoke tests pass.

SMOKE_TEST = False

if SMOKE_TEST:

    if len(SELECTED_EXAMPLES) < 2:
        raise RuntimeError(
            "Smoke mode requires at least two selected benchmark records."
        )

    INFERENCE_MODE = "smoke_test"
    INFERENCE_EXAMPLES = SELECTED_EXAMPLES[:2]

    INFERENCE_DIR = RUN_DIR / "inference_smoke_test"
    INFERENCE_DIR.mkdir(parents=True, exist_ok=False)

    OUTPUT_TAG = "smoke_test_"

else:

    INFERENCE_MODE = "full"
    INFERENCE_EXAMPLES = SELECTED_EXAMPLES

    INFERENCE_DIR = RUN_DIR
    OUTPUT_TAG = ""


# ------------------------------------------------------------
# 6. Load C0 or construct C1
# ------------------------------------------------------------

def load_condition(name):

    if name not in {"C0", "C1"}:
        raise ValueError(f"Unknown model condition: {name}")

    started = time.perf_counter()

    # FP16 is used for compatibility with Tesla T4.
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        revision=MODEL_REVISION,
        trust_remote_code=False,
        torch_dtype=torch.float16,
        low_cpu_mem_usage=True,
        device_map={"": "cuda:0"},
        attn_implementation=ATTN_IMPL,
    )

    model.eval()

    model.config.use_cache = False
    model.generation_config.use_cache = False

    resolved_commit = getattr(model.config, "_commit_hash", None)

    if (
        resolved_commit is not None
        and resolved_commit != MODEL_REVISION
    ):
        raise RuntimeError(
            f"{name} checkpoint commit mismatch: "
            f"{resolved_commit!r} != {MODEL_REVISION!r}"
        )

    if (
        int(model.config.hidden_size)
        != int(model_config.hidden_size)
    ):
        raise RuntimeError(
            f"{name} hidden size differs from the pinned "
            "model architecture configuration."
        )

    if len(model.model.layers) != 28:
        raise RuntimeError(
            f"Fresh {name} source checkpoint must have 28 decoder layers."
        )

    # Verify FP16 loading.
    if model.get_input_embeddings().weight.dtype != torch.float16:
        raise RuntimeError(
            f"{name} model weights were not loaded in float16."
        )

    # --------------------------------------------------------
    # Rebuild C1 from original pinned model + frozen A/b
    # --------------------------------------------------------

    if name == "C1":

        original_classes = [
            type(layer).__name__
            for layer in model.model.layers
        ]

        if original_classes[PAIR[0]:PAIR[1] + 1] != [
            "Qwen3DecoderLayer",
            "Qwen3DecoderLayer",
        ]:
            raise RuntimeError(
                "C1 source layers [3,4] are not the original "
                "adjacent Qwen3 decoder blocks."
            )

        _replace_pair(
            model,
            PAIR[0],
            affine["A"],
            affine["b"],
            torch,
        )

        affine_layer = model.model.layers[PAIR[0]]

        if (
            type(affine_layer).__name__ != "AffineQwen3Layer"
            or len(model.model.layers) != 27
        ):
            raise RuntimeError(
                "C1 [3,4] Average surgery did not produce "
                "the expected 27-layer model."
            )

        # Verify deployed affine matrix.
        if not torch.equal(
            affine_layer.weight.detach().cpu(),
            affine["A"].to(dtype=torch.float32),
        ):
            raise RuntimeError(
                "C1 deployed affine matrix differs from "
                "the verified frozen tensor."
            )

        # Verify deployed affine bias.
        if not torch.equal(
            affine_layer.bias.detach().cpu(),
            affine["b"].to(dtype=torch.float32),
        ):
            raise RuntimeError(
                "C1 deployed affine bias differs from "
                "the verified frozen tensor."
            )

    # --------------------------------------------------------
    # 7. Verify final architecture and parameter counts
    # --------------------------------------------------------

    actual_params = nparams(model)

    expected_params = C0_PARAMS if name == "C0" else C1_PARAMS
    expected_layers = 28 if name == "C0" else 27

    if (
        actual_params != expected_params
        or len(model.model.layers) != expected_layers
    ):
        raise RuntimeError(
            f"{name} architecture mismatch: "
            f"parameters={actual_params:,}/{expected_params:,}, "
            f"layers={len(model.model.layers)}/{expected_layers}."
        )

    if (
        model.get_input_embeddings().weight
        is not model.get_output_embeddings().weight
    ):
        raise RuntimeError(
            f"{name} input/output embeddings are not tied as expected."
        )

    device = model.get_input_embeddings().weight.device

    metadata = {
        "condition": name,
        "model_id": MODEL_ID,
        "requested_revision": MODEL_REVISION,
        "resolved_commit": resolved_commit,
        "parameters": actual_params,
        "layers": len(model.model.layers),
        "load_seconds": time.perf_counter() - started,
        "dtype": "float16",
        "attention": ATTN_IMPL,
        "decoding": "greedy",
        "do_sample": False,
        "num_beams": 1,
        "use_cache": False,
        "device": str(device),
    }

    print(
        f"{name} loaded successfully: "
        f"{actual_params:,} parameters, "
        f"{len(model.model.layers)} layers."
    )

    return model, metadata


# ------------------------------------------------------------
# 8. Output parsing
# ------------------------------------------------------------

def parse_mcq(text):

    matches = re.findall(
        r"(?im)^\s*(?:final\s+)?answer\s*[:：]\s*([ABCD])\b",
        text or "",
    )

    return matches[-1].upper() if matches else None


def parse_num(text):

    lines = re.findall(
        r"(?im)^\s*(?:final\s+)?answer\s*[:：]\s*(.*)$",
        text or "",
    )

    numbers = (
        re.findall(
            r"[-+]?(?:\d{1,3}(?:,\d{3})+|\d+)(?:\.\d+)?",
            lines[-1],
        )
        if lines
        else []
    )

    return numbers[-1].replace(",", "") if numbers else None


def score(task_type, prediction, gold):

    if prediction is None:
        return False

    if task_type == "mcq":
        return str(prediction).upper() == str(gold).upper()

    try:
        return Decimal(prediction) == Decimal(str(gold))

    except (InvalidOperation, ValueError):
        return False


# ------------------------------------------------------------
# 9. Run inference for one condition
# ------------------------------------------------------------

def run_condition(name):

    prediction_path = (
        INFERENCE_DIR / f"{OUTPUT_TAG}predictions_{name}.jsonl"
    )

    error_path = (
        INFERENCE_DIR / f"{OUTPUT_TAG}errors_{name}.jsonl"
    )

    if prediction_path.exists() or error_path.exists():
        raise FileExistsError(
            f"Refusing to mix or overwrite existing {name} "
            f"outputs in {INFERENCE_DIR}."
        )

    prediction_partial = Path(str(prediction_path) + ".partial")
    error_partial = Path(str(error_path) + ".partial")

    model, meta = load_condition(name)

    device = model.get_input_embeddings().weight.device

    try:

        torch.cuda.reset_peak_memory_stats(device)

        # One warm-up generation before timed inference.
        warm_ids, warm_mask = encode_chat(
            "Reply exactly: Answer: A"
        )

        with torch.inference_mode():

            model.generate(
                input_ids=warm_ids.to(device),
                attention_mask=warm_mask.to(device),
                max_new_tokens=1,
                do_sample=False,
                num_beams=1,
                use_cache=False,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        torch.cuda.synchronize(device)

        print(f"{name}: warm-up completed. Starting predictions.")

        counters = defaultdict(
            lambda: {
                "selected": 0,
                "correct": 0,
                "errors": 0,
                "payload_tokens": 0,
                "prompt_tokens": 0,
                "generated_tokens": 0,
                "generation_seconds": 0.0,
            }
        )

        total_started = time.perf_counter()
        written = 0

        with (
            prediction_partial.open(
                "w",
                encoding="utf-8",
                newline="\n",
            ) as out,
            error_partial.open(
                "w",
                encoding="utf-8",
                newline="\n",
            ) as err,
        ):

            for example_index, (task, row) in enumerate(
                INFERENCE_EXAMPLES,
                start=1,
            ):

                ids, mask = encode_chat(row["prompt"])

                input_hash = digest_ids(ids)
                prompt_hash = digest_text(row["prompt"])

                if (
                    input_hash != row["input_ids_sha256"]
                    or prompt_hash != row["prompt_sha256"]
                ):
                    raise RuntimeError(
                        f"Prompt/tokenizer hash mismatch: "
                        f"{row['example_id']}"
                    )

                task_type = (
                    "mgsm"
                    if task["dataset"] == "MGSM"
                    else "mcq"
                )

                result = {
                    "dataset": task["dataset"],
                    "config": task["config"],
                    "example_id": row["example_id"],
                    "question": row["question"],
                    "choices": row.get("choices"),
                    "subject": row.get("subject"),
                    "condition": name,
                    "run_mode": INFERENCE_MODE,
                    "model_id": MODEL_ID,
                    "model_revision": MODEL_REVISION,
                    "gold": row["gold"],
                    "fewshot_ids": row["fewshot_ids"],
                    "payload_tokens": int(row["payload_tokens"]),
                    "input_tokens": int(ids.shape[1]),
                    "prompt_tokens": int(ids.shape[1]),
                    "generated_tokens": 0,
                    "output_tokens": 0,
                    "total_tokens": int(ids.shape[1]),
                    "prompt_sha256": prompt_hash,
                    "input_ids_sha256": input_hash,
                    "status": "error",
                    "raw_output": "",
                    "prediction": None,
                    "correct": False,
                    "error_type": None,
                    "error_message": None,
                    "generation_seconds": 0.0,
                }

                key = task["dataset"] + "|" + task["config"]
                counts = counters[key]

                counts["selected"] += 1
                counts["payload_tokens"] += result["payload_tokens"]
                counts["prompt_tokens"] += result["prompt_tokens"]

                generation_started = time.perf_counter()
                generated_ids = None

                try:

                    ids = ids.to(device)
                    mask = mask.to(device)

                    with torch.inference_mode():

                        generated = model.generate(
                            input_ids=ids,
                            attention_mask=mask,
                            max_new_tokens=int(row["max_new_tokens"]),
                            do_sample=False,
                            num_beams=1,
                            use_cache=False,
                            pad_token_id=tokenizer.eos_token_id,
                            eos_token_id=tokenizer.eos_token_id,
                            repetition_penalty=1.0,
                            length_penalty=1.0,
                        )

                    generated_ids = generated[0, ids.shape[1]:]

                    result["generated_tokens"] = int(
                        generated_ids.numel()
                    )

                    result["output_tokens"] = result["generated_tokens"]

                    result["total_tokens"] = (
                        result["input_tokens"]
                        + result["output_tokens"]
                    )

                    torch.cuda.synchronize(device)

                    raw_output = tokenizer.decode(
                        generated_ids,
                        skip_special_tokens=True,
                    )

                    prediction = (
                        parse_num(raw_output)
                        if task_type == "mgsm"
                        else parse_mcq(raw_output)
                    )

                    correct = score(
                        task_type,
                        prediction,
                        row["gold"],
                    )

                    result.update({
                        "raw_output": raw_output,
                        "prediction": prediction,
                        "correct": bool(correct),
                        "status": "ok",
                    })

                except Exception as exc:

                    result.update({
                        "correct": False,
                        "status": "error",
                        "error_type": type(exc).__name__,
                        "error_message": str(exc),
                        "traceback": traceback.format_exc()[-5000:],
                    })

                    if "out of memory" in str(exc).lower():
                        torch.cuda.empty_cache()

                finally:

                    result["generation_seconds"] = (
                        time.perf_counter() - generation_started
                    )

                counts["generated_tokens"] += result["generated_tokens"]

                counts["generation_seconds"] += (
                    result["generation_seconds"]
                )

                if result["status"] == "error":

                    counts["errors"] += 1

                    err.write(
                        json.dumps(result, ensure_ascii=False) + "\n"
                    )
                    err.flush()

                else:

                    counts["correct"] += int(result["correct"])

                out.write(
                    json.dumps(result, ensure_ascii=False) + "\n"
                )
                out.flush()

                written += 1

                print(
                    f"{name}: completed example "
                    f"{example_index}/{len(INFERENCE_EXAMPLES)} "
                    f"({row['example_id']})"
                )

        if written != len(INFERENCE_EXAMPLES):
            raise RuntimeError(
                f"{name} wrote {written} records; "
                f"expected {len(INFERENCE_EXAMPLES)}."
            )

        torch.cuda.synchronize(device)

        prediction_partial.replace(prediction_path)
        error_partial.replace(error_path)

        meta.update({
            "run_mode": INFERENCE_MODE,
            "inference_seconds": time.perf_counter() - total_started,
            "peak_allocated_gib": (
                torch.cuda.max_memory_allocated(device) / (1024 ** 3)
            ),
            "peak_reserved_gib": (
                torch.cuda.max_memory_reserved(device) / (1024 ** 3)
            ),
            "per_language": dict(counters),
            "prediction_file": prediction_path.name,
            "error_file": error_path.name,
            "prediction_records": written,
            "expected_prediction_records": len(INFERENCE_EXAMPLES),
            "warmups": 1,
        })

        print(
            f"{name}: inference completed. "
            f"Predictions: {written}, "
            f"errors: {sum(x['errors'] for x in counters.values())}"
        )

        return meta

    finally:

        del model
        gc.collect()
        torch.cuda.empty_cache()


# ------------------------------------------------------------
# 10. Run C0 first, then C1
# ------------------------------------------------------------

print("Starting matched inference.")
print("Mode:", INFERENCE_MODE)
print("Examples per condition:", len(INFERENCE_EXAMPLES))
print("Precision: float16")
print("Decoding: greedy")

system = {
    "C0": run_condition("C0"),
    "C1": run_condition("C1"),
}

system["run_mode"] = INFERENCE_MODE
system["prediction_records_per_condition"] = len(INFERENCE_EXAMPLES)

system_filename = f"{OUTPUT_TAG}model_system_metrics.json"

(INFERENCE_DIR / system_filename).write_text(
    json.dumps(system, indent=2, sort_keys=True),
    encoding="utf-8",
)

if SMOKE_TEST:

    print(
        "Two-record inference smoke test completed for C0 and C1:",
        INFERENCE_DIR.resolve(),
    )

    print(
        "Smoke outputs are labeled and isolated; "
        "paired benchmark scoring is disabled in this mode."
    )

else:

    print(
        "C0 and C1 full prediction files are complete:",
        INFERENCE_DIR.resolve(),
    )


Model configuration: Qwen3Config
Model hidden size: 2048
Expected C0 parameters: 1,720,574,976
Expected C1 parameters: 1,624,099,328
Starting matched inference.
Mode: full
Examples per condition: 31220
Precision: float16
Decoding: greedy


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

C0 loaded successfully: 1,720,574,976 parameters, 28 layers.
C0: warm-up completed. Starting predictions.
C0: completed example 1/31220 (MMMLU:AR_XY:test:4228)
C0: completed example 2/31220 (MMMLU:AR_XY:test:12689)
C0: completed example 3/31220 (MMMLU:AR_XY:test:9232)
C0: completed example 4/31220 (MMMLU:AR_XY:test:4372)
C0: completed example 5/31220 (MMMLU:AR_XY:test:2087)
C0: completed example 6/31220 (MMMLU:AR_XY:test:12323)
C0: completed example 7/31220 (MMMLU:AR_XY:test:11420)
C0: completed example 8/31220 (MMMLU:AR_XY:test:4833)
C0: completed example 9/31220 (MMMLU:AR_XY:test:728)
C0: completed example 10/31220 (MMMLU:AR_XY:test:13875)
C0: completed example 11/31220 (MMMLU:AR_XY:test:1427)
C0: completed example 12/31220 (MMMLU:AR_XY:test:13416)
C0: completed example 13/31220 (MMMLU:AR_XY:test:3991)
C0: completed example 14/31220 (MMMLU:AR_XY:test:13519)
C0: completed example 15/31220 (MMMLU:AR_XY:test:4499)
C0: completed example 16/31220 (MMMLU:AR_XY:test:11107)
C0: completed exa

KeyboardInterrupt: 

In [ ]:
from pathlib import Path
import json

print("RUN DIRECTORY:")
print(RUN_DIR.resolve())

prediction_file = RUN_DIR / "predictions_C0.jsonl.partial"
error_file = RUN_DIR / "errors_C0.jsonl.partial"

for path in (prediction_file, error_file):

    print("\n" + "=" * 55)
    print("FILE:", path.name)
    print("EXISTS:", path.exists())

    if not path.exists():
        continue

    print("SIZE (MB):", round(path.stat().st_size / (1024 * 1024), 3))

    valid_records = 0
    invalid_lines = 0
    last_valid_record = None

    with path.open("r", encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            if not line.strip():
                continue

            try:
                record = json.loads(line)
                valid_records += 1
                last_valid_record = record

            except json.JSONDecodeError:
                invalid_lines += 1

    print("Valid records:", valid_records)
    print("Invalid/incomplete lines:", invalid_lines)

    if last_valid_record is not None:
        print("Last valid example ID:", last_valid_record.get("example_id"))
        print("Last valid status:", last_valid_record.get("status"))

RUN DIRECTORY:
/content/benchmark_results/20261106T092234661371Z

FILE: predictions_C0.jsonl.partial
EXISTS: True
SIZE (MB): 1.783
Valid records: 1090
Invalid/incomplete lines: 0
Last valid example ID: MMMLU:DE_DE:test:12367
Last valid status: ok

FILE: errors_C0.jsonl.partial
EXISTS: True
SIZE (MB): 0.0
Valid records: 0
Invalid/incomplete lines: 0


In [ ]:
from pathlib import Path
import json
import torch

print("=" * 55)
print("GPU STATUS")
print("=" * 55)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU currently attached.")

print("\n" + "=" * 55)
print("CHECKING PREVIOUS BENCHMARK FILES")
print("=" * 55)

base = Path("/content/benchmark_results")

if not base.exists():
    print("benchmark_results directory does not exist.")
else:
    for run_dir in sorted(base.iterdir()):
        if not run_dir.is_dir():
            continue

        print("\nRUN:", run_dir.name)

        for filename in [
            "predictions_C0.jsonl.partial",
            "errors_C0.jsonl.partial",
            "predictions_C0.jsonl",
            "predictions_C1.jsonl.partial",
            "predictions_C1.jsonl",
        ]:
            path = run_dir / filename

            if path.exists():
                count = 0
                with path.open("r", encoding="utf-8") as f:
                    for line in f:
                        if line.strip():
                            json.loads(line)
                            count += 1

                print(f"  {filename}: {count} records")

GPU STATUS
CUDA available: False
No GPU currently attached.

CHECKING PREVIOUS BENCHMARK FILES
benchmark_results directory does not exist.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

DRIVE = Path("/content/drive/MyDrive")

search_names = [
    "benchmark_results",
    "predictions_C0.jsonl.partial",
    "predictions_C0.jsonl",
    "c1_frozen_artifact.zip",
    "frozen_c1_artifact_record.json",
]

print("Searching Google Drive...\n")

found = []

for path in DRIVE.rglob("*"):
    if path.name in search_names:
        found.append(path)
        print(path)

if not found:
    print("No matching benchmark files found in Google Drive.")

Searching Google Drive...

No matching benchmark files found in Google Drive.


In [ ]:
# ============================================================
# PAIRED BENCHMARK SCORING AND FINAL RUN MANIFEST
# ============================================================

# Smoke-test artifacts are deliberately excluded from full benchmark scoring.
if globals().get("INFERENCE_MODE") == "smoke_test":
    raise RuntimeError(
        "Paired benchmark scoring is disabled after a two-record smoke test. "
        "Set SMOKE_TEST = False in Cell 8 and rerun full inference "
        "to create benchmark scores."
    )


# ------------------------------------------------------------
# 1. IMPORTS
# ------------------------------------------------------------

import csv
import json
import sys
from pathlib import Path
from collections import defaultdict
from importlib import metadata


# ------------------------------------------------------------
# 2. VERIFY FULL INFERENCE COMPLETED
# ------------------------------------------------------------

if globals().get("INFERENCE_MODE") != "full":
    raise RuntimeError(
        "Inference mode is not 'full'. "
        "Run the complete C0/C1 benchmark inference before scoring."
    )

if "system" not in globals():
    raise RuntimeError(
        "System metrics are missing. Run the full inference cell first."
    )

if "C0" not in system or "C1" not in system:
    raise RuntimeError(
        "C0 or C1 system metrics are missing."
    )

if "C0_PARAMS" not in globals() or "C1_PARAMS" not in globals():
    raise RuntimeError(
        "Model parameter counts are missing."
    )

if system.get("run_mode") != "full":
    raise RuntimeError(
        f"System metrics are not from a full run: {system.get('run_mode')!r}"
    )

if system.get("prediction_records_per_condition") != len(
    [row for task in tasks for row in task["selected"]]
):
    raise RuntimeError(
        "System metrics prediction count does not match the selected benchmark records."
    )

for condition in ("C0", "C1"):
    if system[condition].get("run_mode") != "full":
        raise RuntimeError(
            f"{condition} system metrics do not describe a full inference run."
        )

    if system[condition].get("dtype") != "float16":
        raise RuntimeError(
            f"{condition} inference dtype is not float16: "
            f"{system[condition].get('dtype')!r}"
        )


# ------------------------------------------------------------
# 3. VERIFY FROZEN MODEL CONFIGURATION
# ------------------------------------------------------------

if PAIR != [3, 4] or OPERATOR != "Average":
    raise RuntimeError(
        f"Expected frozen pair [3,4]/Average; found {PAIR}/{OPERATOR}."
    )

if C0_PARAMS != 1_720_574_976:
    raise RuntimeError(
        f"Unexpected C0 parameter count: {C0_PARAMS:,}"
    )

if C1_PARAMS != 1_624_099_328:
    raise RuntimeError(
        f"Unexpected C1 parameter count: {C1_PARAMS:,}"
    )

if artifact_record["manifest"].get("selection_metric") != SELECTION_METRIC:
    raise RuntimeError(
        "Frozen C1 artifact selection metric does not match the benchmark protocol."
    )

selected_artifact = artifact_record["selected_c1"]

if (
    selected_artifact.get("pair_layers_zero_based") != PAIR
    or selected_artifact.get("operator") != OPERATOR
):
    raise RuntimeError(
        "Frozen C1 artifact does not match the [3,4] Average configuration."
    )


# ------------------------------------------------------------
# 4. READ JSONL PREDICTION FILES
# ------------------------------------------------------------

def read_jsonl(path):

    with Path(path).open(encoding="utf-8") as f:

        for line_number, line in enumerate(f, start=1):

            if line.strip():

                try:
                    yield json.loads(line)

                except json.JSONDecodeError as exc:

                    raise ValueError(
                        f"Invalid JSON in {Path(path).name}:{line_number}: {exc}"
                    ) from exc


# Freeze the expected benchmark record order.
expected_examples = [
    (task, row)
    for task in tasks
    for row in task["selected"]
]

expected_ids = [
    row["example_id"]
    for _, row in expected_examples
]

if len(expected_ids) != len(set(expected_ids)):
    raise RuntimeError(
        "Selected task records contain duplicate example IDs."
    )

if not expected_ids:
    raise RuntimeError(
        "No selected examples exist; paired scoring is not valid."
    )


# ------------------------------------------------------------
# 5. VALIDATE EACH COMPLETE PREDICTION FILE
# ------------------------------------------------------------

def validate_prediction_file(path, condition):

    if not path.is_file():
        raise FileNotFoundError(
            f"Required {condition} predictions are missing: {path}"
        )

    records = list(read_jsonl(path))

    if len(records) != len(expected_examples):
        raise RuntimeError(
            f"{condition} prediction count mismatch: "
            f"found {len(records)}, expected {len(expected_examples)}."
        )

    for index, ((task, selected), record) in enumerate(
        zip(expected_examples, records)
    ):

        expected_id = selected["example_id"]

        # Verify record ordering.
        if record.get("example_id") != expected_id:
            raise RuntimeError(
                f"{condition} record {index} is "
                f"{record.get('example_id')!r}; "
                f"expected {expected_id!r}."
            )

        # Verify all frozen input and benchmark metadata.
        expected_fields = {
            "dataset": task["dataset"],
            "config": task["config"],
            "question": selected["question"],
            "choices": selected.get("choices"),
            "subject": selected.get("subject"),
            "gold": selected["gold"],
            "fewshot_ids": selected["fewshot_ids"],
            "payload_tokens": int(selected["payload_tokens"]),
            "input_tokens": int(selected["input_tokens"]),
            "prompt_tokens": int(selected["input_tokens"]),
            "prompt_sha256": selected["prompt_sha256"],
            "input_ids_sha256": selected["input_ids_sha256"],
            "condition": condition,
            "run_mode": "full",
            "model_id": MODEL_ID,
            "model_revision": MODEL_REVISION,
        }

        for field, expected in expected_fields.items():

            if record.get(field) != expected:

                raise RuntimeError(
                    f"{condition} prediction {expected_id} "
                    f"has mismatched {field}: "
                    f"{record.get(field)!r} != {expected!r}."
                )

        # Verify generation status.
        if record.get("status") not in {"ok", "error"}:
            raise RuntimeError(
                f"{condition} prediction {expected_id} has invalid status."
            )

        # Correctness must always be a Boolean.
        if not isinstance(record.get("correct"), bool):
            raise RuntimeError(
                f"{condition} prediction {expected_id} "
                "lacks boolean correctness."
            )

        # Generation errors must count as incorrect.
        if (
            record["status"] == "error"
            and record["correct"] is not False
        ):
            raise RuntimeError(
                f"Generation error was not retained as incorrect: "
                f"{condition}/{expected_id}."
            )

        # Verify token-count fields.
        for field in (
            "generated_tokens",
            "output_tokens",
            "total_tokens",
        ):

            if (
                not isinstance(record.get(field), int)
                or isinstance(record.get(field), bool)
                or record[field] < 0
            ):
                raise RuntimeError(
                    f"{condition} prediction {expected_id} "
                    f"has invalid {field}."
                )

        if record["generated_tokens"] != record["output_tokens"]:
            raise RuntimeError(
                f"{condition} prediction {expected_id} "
                "output-token counts differ."
            )

        if (
            record["total_tokens"]
            != record["input_tokens"] + record["output_tokens"]
        ):
            raise RuntimeError(
                f"{condition} prediction {expected_id} "
                "total-token count differs."
            )

        # A successful generation must not carry an error message.
        if record["status"] == "ok" and record.get("error_type") is not None:
            raise RuntimeError(
                f"{condition} successful prediction {expected_id} "
                "unexpectedly contains an error type."
            )

    return records


# Validate C0 and C1 independently.
predictions_c0 = validate_prediction_file(
    RUN_DIR / "predictions_C0.jsonl",
    "C0",
)

predictions_c1 = validate_prediction_file(
    RUN_DIR / "predictions_C1.jsonl",
    "C1",
)


# ------------------------------------------------------------
# 6. VERIFY C0/C1 RECORDS ARE PAIRED CORRECTLY
# ------------------------------------------------------------

for index, (a, b) in enumerate(
    zip(predictions_c0, predictions_c1)
):

    for field in (
        "dataset",
        "config",
        "example_id",
        "question",
        "choices",
        "subject",
        "gold",
        "fewshot_ids",
        "payload_tokens",
        "input_tokens",
        "prompt_tokens",
        "prompt_sha256",
        "input_ids_sha256",
    ):

        if a.get(field) != b.get(field):

            raise RuntimeError(
                f"C0/C1 paired mismatch at record {index}, "
                f"field {field}."
            )

if len(predictions_c0) != len(predictions_c1):
    raise RuntimeError(
        "C0/C1 result counts differ."
    )

print(
    f"Validated {len(predictions_c0):,} C0 predictions "
    f"and {len(predictions_c1):,} C1 predictions."
)

print(
    "C0/C1 record order, IDs, prompts, hashes, and metadata match."
)


# ------------------------------------------------------------
# 7. CREATE PAIRED PREDICTIONS
# ------------------------------------------------------------

totals = defaultdict(
    lambda: {
        "n": 0,
        "correct0": 0,
        "correct1": 0,
        "errors0": 0,
        "errors1": 0,
        "payload": 0,
        "prompt": 0,
    }
)

paired_path = RUN_DIR / "paired_predictions.jsonl"

if paired_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing paired predictions: {paired_path}"
    )

with paired_path.open("w", encoding="utf-8") as out:

    for a, b in zip(predictions_c0, predictions_c1):

        key = a["dataset"] + "|" + a["config"]

        total = totals[key]

        total["n"] += 1
        total["payload"] += a["payload_tokens"]
        total["prompt"] += a["prompt_tokens"]

        total["correct0"] += int(a["correct"])
        total["correct1"] += int(b["correct"])

        total["errors0"] += int(a["status"] == "error")
        total["errors1"] += int(b["status"] == "error")

        row = {
            "dataset": a["dataset"],
            "config": a["config"],
            "example_id": a["example_id"],
            "question": a["question"],
            "choices": a.get("choices"),
            "subject": a.get("subject"),
            "gold": a["gold"],
            "fewshot_ids": a["fewshot_ids"],
            "payload_tokens": a["payload_tokens"],
            "prompt_tokens": a["prompt_tokens"],
            "prompt_sha256": a["prompt_sha256"],
            "input_ids_sha256": a["input_ids_sha256"],

            "C0": {
                k: a.get(k)
                for k in (
                    "status",
                    "raw_output",
                    "prediction",
                    "correct",
                    "error_type",
                    "error_message",
                    "generated_tokens",
                    "output_tokens",
                    "total_tokens",
                )
            },

            "C1": {
                k: b.get(k)
                for k in (
                    "status",
                    "raw_output",
                    "prediction",
                    "correct",
                    "error_type",
                    "error_message",
                    "generated_tokens",
                    "output_tokens",
                    "total_tokens",
                )
            },

            "accuracy_delta_C1_minus_C0": (
                int(b["correct"]) - int(a["correct"])
            ),
        }

        out.write(
            json.dumps(row, ensure_ascii=False) + "\n"
        )


# ------------------------------------------------------------
# 8. CREATE PER-LANGUAGE SUMMARY
# ------------------------------------------------------------

summary = []

for task in tasks:

    key = task["dataset"] + "|" + task["config"]

    total = totals[key]

    n = total["n"]

    accuracy0 = (
        total["correct0"] / n
        if n
        else None
    )

    accuracy1 = (
        total["correct1"] / n
        if n
        else None
    )

    summary.append({

        "dataset": task["dataset"],
        "language_config": task["config"],
        "split": "test",

        "source_questions": task["source_test_rows"],
        "selected_questions": task["selected_questions"],

        "unique_test_payload_tokens": task["payload_tokens"],
        "target_tokens": TOKEN_TARGET,
        "target_met": task["target_met"],
        "shortfall_tokens": task["shortfall_tokens"],

        "full_prompt_input_tokens_per_model": total["prompt"],

        "skipped_examples": task["skipped_count"],

        "C0_correct": total["correct0"],
        "C0_errors_counted_wrong": total["errors0"],
        "C0_accuracy": accuracy0,

        "C1_correct": total["correct1"],
        "C1_errors_counted_wrong": total["errors1"],
        "C1_accuracy": accuracy1,

        "accuracy_delta_C1_minus_C0": (
            accuracy1 - accuracy0
            if n
            else None
        ),

        "matched_hashes_verified": True,
    })


if not summary:
    raise RuntimeError(
        "Per-language summary is empty; refusing to create final outputs."
    )


summary_csv_path = RUN_DIR / "per_language_summary.csv"
summary_json_path = RUN_DIR / "per_language_summary.json"

for path in (summary_csv_path, summary_json_path):
    if path.exists():
        raise FileExistsError(
            f"Refusing to overwrite existing summary file: {path}"
        )


with summary_csv_path.open(
    "w",
    newline="",
    encoding="utf-8",
) as f:

    writer = csv.DictWriter(
        f,
        fieldnames=list(summary[0].keys()),
    )

    writer.writeheader()
    writer.writerows(summary)


summary_json_path.write_text(
    json.dumps(
        summary,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 9. CREATE DATASET COVERAGE SUMMARY
# ------------------------------------------------------------

coverage_summary = {

    "MMMLU": {
        "configured_config_count": len(MMMLU_CONFIGS),

        "evaluated_config_count": sum(
            task["dataset"] == "MMMLU"
            for task in tasks
        ),
    },

    "INCLUDE": {
        **INCLUDE_COVERAGE_SUMMARY,

        "evaluated_configs": INCLUDE_CONFIGS,

        "C0_and_C1_use_same_configs": True,
    },

    "MGSM": {
        "configured_config_count": len(MGSM_CONFIGS),

        "evaluated_config_count": sum(
            task["dataset"] == "MGSM"
            for task in tasks
        ),
    },
}


coverage_path = RUN_DIR / "coverage_summary.json"

if coverage_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing coverage summary: {coverage_path}"
    )

coverage_path.write_text(
    json.dumps(
        coverage_summary,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


print(
    "INCLUDE coverage recorded in run outputs:",
    coverage_summary["INCLUDE"]["coverage"],
)


# ------------------------------------------------------------
# 10. VERIFY SYSTEM METRICS BEFORE FINAL MANIFEST
# ------------------------------------------------------------

for condition in ("C0", "C1"):

    if "parameters" not in system[condition]:
        raise RuntimeError(
            f"{condition} parameter metrics are missing."
        )

    if system[condition]["parameters"] != (
        C0_PARAMS if condition == "C0" else C1_PARAMS
    ):
        raise RuntimeError(
            f"{condition} parameter count in system metrics is incorrect."
        )

    if system[condition].get("prediction_records") != len(expected_ids):
        raise RuntimeError(
            f"{condition} system prediction count is incorrect."
        )

    if system[condition].get("dtype") != "float16":
        raise RuntimeError(
            f"{condition} system dtype does not match FP16 inference."
        )


# ------------------------------------------------------------
# 11. CREATE FINAL RUN MANIFEST
# ------------------------------------------------------------

manifest = {

    "run_id_utc": RUN_ID,

    "model": {
        "id": MODEL_ID,
        "revision": MODEL_REVISION,

        "pair": PAIR,
        "operator": OPERATOR,

        "selection_metric": SELECTION_METRIC,

        "C0_parameters": C0_PARAMS,
        "C1_parameters": C1_PARAMS,
    },

    "datasets": {

        "MMMLU": {
            "id": MMMLU_ID,
            "revision": MMMLU_REV,
            "configs": MMMLU_CONFIGS,

            "fewshot_source": (
                MMLU_DEV_ID
                + "/all/dev@"
                + MMLU_DEV_REV
            ),

            "shots": 5,
        },

        "INCLUDE": {
            "id": INCLUDE_ID,
            "revision": INCLUDE_REV,
            "configs": INCLUDE_CONFIGS,

            "configured_config_count": len(INCLUDE_ALL_CONFIGS),

            "evaluated_config_count": len(INCLUDE_CONFIGS),

            "coverage": INCLUDE_COVERAGE_SUMMARY["coverage"],

            "excluded_configs": INCLUDE_EXCLUDED_CONFIGS,

            "fewshot_source": "same config validation first five",

            "shots": 5,
        },

        "MGSM": {
            "id": MGSM_ID,
            "revision": MGSM_REV,
            "configs": MGSM_CONFIGS,

            "fewshot_source": "same config train all eight",

            "shots": 8,
        },
    },

    "coverage": coverage_summary,

    "selected_prediction_records_per_model": len(expected_ids),

    "prediction_files": {
        "C0": "predictions_C0.jsonl",
        "C1": "predictions_C1.jsonl",
    },

    "token_target_unique_test_payload_per_language": TOKEN_TARGET,

    "selection": (
        "MMMLU/INCLUDE: stable SHA-256 order without replacement "
        "through target; MGSM: evaluate every available test row exactly once"
    ),

    "tokenizer": {
        "model_id": MODEL_ID,
        "requested_revision": MODEL_REVISION,
        "resolved_snapshot_commit": TOKENIZER_RESOLVED_COMMIT,
    },

    "evaluation": {

        # IMPORTANT: This run uses FP16, not BF16.
        "dtype": "float16",

        "attention": ATTN_IMPL,

        "do_sample": False,
        "num_beams": 1,
        "use_cache": False,
        "enable_thinking": False,

        "mcq_max_new_tokens": MCQ_MAX_NEW_TOKENS,
        "mgsm_max_new_tokens": MGSM_MAX_NEW_TOKENS,

        "seed": SEED,

        "precision_deviation": (
            "FP16 used instead of BF16 for Tesla T4 compatibility."
        ),
    },

    "runtime": {
        "python": sys.version.split()[0],

        "torch": torch.__version__,

        "cuda": torch.version.cuda,

        "gpu": torch.cuda.get_device_name(0),

        "transformers": metadata.version("transformers"),

        "datasets": metadata.version("datasets"),
    },

    "frozen_c1_artifact": artifact_record,

    "system_metrics": system,

    "test_data_used_for_c1_selection": False,

    "cgos_rerun": False,
}


manifest_path = RUN_DIR / "run_manifest.json"

if manifest_path.exists():
    raise FileExistsError(
        f"Refusing to overwrite existing run manifest: {manifest_path}"
    )


manifest_path.write_text(
    json.dumps(
        manifest,
        ensure_ascii=False,
        indent=2,
        sort_keys=True,
    ),
    encoding="utf-8",
)


# ------------------------------------------------------------
# 12. FINAL CONFIRMATION
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PAIRED BENCHMARK SCORING COMPLETED SUCCESSFULLY")
print("=" * 65)

print(f"Run directory: {RUN_DIR.resolve()}")

print(f"Selected examples per model: {len(expected_ids):,}")

print(f"C0 parameters: {C0_PARAMS:,}")
print(f"C1 parameters: {C1_PARAMS:,}")

print(f"C0 prediction records: {len(predictions_c0):,}")
print(f"C1 prediction records: {len(predictions_c1):,}")

print("Evaluation dtype: FP16")
print("C1 configuration: layers [3,4], Average")
print("CGOS rerun: False")
print("Test data used for C1 selection: False")

print("\nCreated output files:")
print("1. paired_predictions.jsonl")
print("2. per_language_summary.csv")
print("3. per_language_summary.json")
print("4. coverage_summary.json")
print("5. run_manifest.json")

print("\nBoth prediction streams contain one verified record per selected example.")
print("All paired records passed the input and hash validation checks.")

RuntimeError: System metrics are not from a full run: 'smoke_test'

## Risks and status

- The affine replacement omits attention, RoPE, and KV-cache updates. This is an architectural limitation, not something the benchmark notebook can correct.
- MMMLU has no in-language dev split in the pinned release; its English MMLU dev exemplars are a proposed cross-language compromise. An approved translated non-test exemplar set is needed if in-language examples are required.
- INCLUDE is pinned to original include-base-44. Urdu, Uzbek, and Vietnamese are excluded because every validation row has choices=null and no alternative non-test split exists; reported INCLUDE coverage is 41/44. The two dataset versions must not be mixed.
- Prompt wording, shot counts, answer parsers, output limits, and the token accounting interpretation are proposed because the assignment handout does not define them. Freeze or obtain mentor approval before viewing test scores.
- No benchmark has been executed. Stage the recovered artifact in `c1_frozen_artifact/`; the artifact gate stops before benchmark test data is opened if the files are absent or fail verification.